# 02 - Fashion CNN V0 - Inspection dataset

Objectif : preparer Fashion Product Images Small pour un futur CNN V0 sans entrainer de modele dans ce notebook.

La cible future est `product_type_v0`, derivee de `styles.csv.articleType`. Le modele predira un type produit visible, puis `canonical_category` sera derivee pour le moteur outfit.

Ce notebook inspecte le dataset et verifie la configuration validee. L'entrainement reste volontairement separe.


## 1. Monter Google Drive


In [ ]:
from pathlib import Path

from google.colab import drive

DRIVE_MOUNT = Path('/content/drive')
if (DRIVE_MOUNT / 'MyDrive').exists():
    print('Google Drive deja monte.')
else:
    drive.mount(str(DRIVE_MOUNT))

DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive'
print(f'Drive root: {DRIVE_ROOT}')


## 2. Cloner ou mettre a jour le repo GitHub


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/MilFhey/fit-outfit-advisor.git'
BRANCH = 'main'
REPO_DIR = Path('/content/fit-outfit-advisor-repo')
PROJECT_DIR = REPO_DIR / 'fit-outfit-advisor'

if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    print('Repo existant : mise a jour.')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=str(REPO_DIR), check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=str(REPO_DIR), check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=str(REPO_DIR), check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'Dossier projet absent dans le repo clone : {PROJECT_DIR}')

sys.path.insert(0, str(PROJECT_DIR))
print(f'Repo pret : {REPO_DIR}')
print(f'Projet pret : {PROJECT_DIR}')


## 3. Installer les dependances


In [ ]:
requirements_path = PROJECT_DIR / 'requirements.txt'
if not requirements_path.exists():
    raise FileNotFoundError(f'Requirements absent : {requirements_path}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'kaggle'], check=True)


## 4. Creer les dossiers temporaires


In [ ]:
RUNTIME_ROOT = Path('/content/fit-outfit-runtime')
KAGGLE_DOWNLOAD_DIR = RUNTIME_ROOT / 'kaggle_downloads'
CONTENT_DATA_DIR = RUNTIME_ROOT / 'data'
CONTENT_ARTIFACT_DIR = RUNTIME_ROOT / 'artifacts'

for directory in [RUNTIME_ROOT, KAGGLE_DOWNLOAD_DIR, CONTENT_DATA_DIR, CONTENT_ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
    print(directory)


## 5. Charger le Secret Colab Kaggle

Le Secret Colab attendu s'appelle exactement `KAGGLE_API`.


In [ ]:
from google.colab import userdata

kaggle_token = userdata.get('KAGGLE_API')
if not kaggle_token:
    raise ValueError('Secret Colab introuvable : cree ou autorise le Secret KAGGLE_API.')
if not str(kaggle_token).startswith('KGAT_'):
    raise ValueError('Le Secret KAGGLE_API semble invalide : il doit commencer par KGAT_.')

os.environ['KAGGLE_API_TOKEN'] = kaggle_token
os.environ['KAGGLE_API'] = kaggle_token
print('Token Kaggle charge depuis KAGGLE_API.')


## 6. Telecharger Fashion Product Images Small


In [ ]:
KAGGLE_DATASET = 'paramaggarwal/fashion-product-images-small'

existing_files = [path for path in KAGGLE_DOWNLOAD_DIR.rglob('*') if path.is_file()]
if existing_files:
    print(f'Dataset deja present dans {KAGGLE_DOWNLOAD_DIR} ({len(existing_files)} fichiers).')
else:
    subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET, '-p', str(KAGGLE_DOWNLOAD_DIR), '--unzip'],
        check=True,
    )

downloaded_files = sorted(path for path in KAGGLE_DOWNLOAD_DIR.rglob('*') if path.is_file())
print(f'Fichiers detectes : {len(downloaded_files)}')
for path in downloaded_files[:20]:
    print(path.relative_to(KAGGLE_DOWNLOAD_DIR))


## 7. Detecter `styles.csv` et le dossier `images/`


In [ ]:
styles_candidates = sorted(KAGGLE_DOWNLOAD_DIR.rglob('styles.csv'))
if not styles_candidates:
    styles_candidates = sorted(KAGGLE_DOWNLOAD_DIR.rglob('*.csv'))
if not styles_candidates:
    raise FileNotFoundError('Aucun CSV de metadata trouve dans le dataset Kaggle.')

STYLES_CSV = styles_candidates[0]

image_dir_candidates = [path for path in KAGGLE_DOWNLOAD_DIR.rglob('images') if path.is_dir()]
if not image_dir_candidates:
    image_dir_candidates = sorted({path.parent for path in KAGGLE_DOWNLOAD_DIR.rglob('*.jpg')})
if not image_dir_candidates:
    raise FileNotFoundError('Aucun dossier images ou fichier .jpg trouve.')

IMAGE_DIR = image_dir_candidates[0]
print(f'STYLES_CSV = {STYLES_CSV}')
print(f'IMAGE_DIR = {IMAGE_DIR}')
print(f'Nombre de .jpg detectes = {len(list(IMAGE_DIR.glob("*.jpg")))}')


## 8. Inspecter les metadata


In [ ]:
import pandas as pd

df = pd.read_csv(STYLES_CSV, on_bad_lines='skip')
print('shape =', df.shape)
print('colonnes =')
print(list(df.columns))

display(df.head())

missing = (
    df.isna().sum()
    .rename('missing_count')
    .to_frame()
)
missing['missing_pct'] = (missing['missing_count'] / len(df) * 100).round(2)
display(missing.sort_values('missing_count', ascending=False))

for column in ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'usage', 'gender']:
    if column in df.columns:
        print(f'\nDistribution {column}')
        display(df[column].value_counts(dropna=False).head(40).to_frame('count'))


## 9. Charger la configuration de classes et proposer une categorie canonique

Les propositions ci-dessous ne modifient pas `config/fashion_v1_classes.json`. Elles servent uniquement a guider la revue humaine apres inspection.


In [ ]:
import importlib
import json
import sys

PROJECT_DIR_STR = str(PROJECT_DIR)
sys.path = [PROJECT_DIR_STR] + [path for path in sys.path if path != PROJECT_DIR_STR]
for module_name in list(sys.modules):
    if module_name == 'src' or module_name.startswith('src.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

from src.mappings.fashion_v1_mapping import build_article_type_to_product_type_mapping, load_fashion_v1_class_config, map_product_type_to_canonical_category

CLASS_CONFIG_PATH = PROJECT_DIR / 'config' / 'fashion_v1_classes.json'
class_config = load_fashion_v1_class_config(CLASS_CONFIG_PATH)
configured_mapping = build_article_type_to_product_type_mapping(class_config)

DRAFT_ARTICLE_TYPE_PROPOSALS = {
    'Tshirts': 'tshirt', 'Shirts': 'shirt', 'Tops': 'top', 'Kurtas': 'top', 'Kurtis': 'top', 'Tunics': 'top',
    'Jeans': 'jeans', 'Trousers': 'trousers', 'Track Pants': 'trousers', 'Shorts': 'shorts',
    'Dresses': 'dress',
    'Casual Shoes': 'casual_shoes', 'Sports Shoes': 'sports_shoes', 'Formal Shoes': 'dress_shoes', 'Flats': 'dress_shoes',
    'Sandals': 'sandals', 'Flip Flops': 'flip_flops', 'Heels': 'heels',
    'Jackets': 'outerwear', 'Sweaters': 'outerwear', 'Sweatshirts': 'outerwear',
    'Handbags': 'bag', 'Backpacks': 'bag', 'Clutches': 'bag',
    'Watches': 'watch', 'Sunglasses': 'sunglasses',
    'Wallets': 'wallet', 'Belts': 'belt',
    'Earrings': 'jewellery', 'Pendant': 'jewellery', 'Necklace and Chains': 'jewellery',
}

def proposed_product_type(article_type):
    article_type = str(article_type).strip()
    if article_type in configured_mapping:
        return configured_mapping[article_type]
    return DRAFT_ARTICLE_TYPE_PROPOSALS.get(article_type)

def proposed_canonical_category(article_type):
    product_type = proposed_product_type(article_type)
    if product_type is None:
        return None
    return map_product_type_to_canonical_category(product_type, class_config)

print(json.dumps(class_config, indent=2, ensure_ascii=False))
print(f'Nombre articleType configures vers product_type_v0 : {len(configured_mapping)}')


## 10. Verifier images presentes et lisibles, puis produire le tableau final


In [ ]:
from PIL import Image

required_columns = {'id', 'articleType'}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f'Colonnes obligatoires absentes : {sorted(missing_columns)}')

audit_df = df[['id', 'articleType']].copy()
audit_df['articleType'] = audit_df['articleType'].astype(str).str.strip()
audit_df['image_path'] = audit_df['id'].astype(str).str.replace(r'\\.0$', '', regex=True).map(lambda image_id: IMAGE_DIR / f'{image_id}.jpg')
audit_df['image_present'] = audit_df['image_path'].map(Path.exists)

def is_readable_image(path):
    if not path.exists():
        return False
    try:
        with Image.open(path) as image:
            image.verify()
        return True
    except Exception:
        return False

audit_df['image_readable'] = audit_df['image_path'].map(is_readable_image)

summary = audit_df.groupby('articleType').agg(
    metadata_row_count=('articleType', 'size'),
    present_image_count=('image_present', 'sum'),
    readable_image_count=('image_readable', 'sum'),
).reset_index()
summary['proposed_product_type_v0'] = summary['articleType'].map(proposed_product_type)
summary['proposed_canonical_category'] = summary['articleType'].map(proposed_canonical_category)

minimum_count = class_config.get('minimum_readable_images_per_class')
config_status = class_config.get('status')
product_type_readable_counts = (
    summary.dropna(subset=['proposed_product_type_v0'])
    .groupby('proposed_product_type_v0')['readable_image_count']
    .sum()
    .to_dict()
)

def decide(row):
    article_type = row['articleType']
    product_type = row['proposed_product_type_v0']
    if article_type not in configured_mapping or product_type is None:
        return 'exclure', 'articleType absent du mapping product_type_v0 valide'
    if config_status == 'draft_requires_dataset_inspection':
        return 'exclure', 'configuration en brouillon apres inspection requise'
    if minimum_count is None:
        return 'exclure', 'seuil minimal non renseigne'
    product_type_readable_count = int(product_type_readable_counts.get(product_type, 0))
    if product_type_readable_count < int(minimum_count):
        return 'exclure', 'product_type_v0 sous le seuil minimal apres agregation'
    return 'garder', ''

decisions = summary.apply(decide, axis=1, result_type='expand')
summary['decision'] = decisions[0]
summary['exclusion_reason'] = decisions[1]
summary = summary.sort_values(['decision', 'readable_image_count'], ascending=[True, False])

audit_output = CONTENT_ARTIFACT_DIR / 'fashion_v1_article_type_audit.csv'
summary.to_csv(audit_output, index=False)
print(f'Tableau final ecrit : {audit_output}')
display(summary)

print('Comptage images manquantes :', int((~audit_df['image_present']).sum()))
print('Comptage images corrompues/non lisibles :', int((audit_df['image_present'] & ~audit_df['image_readable']).sum()))


## 10 bis. Exporter le rapport complet d'inspection

Cette cellule cree un pack d'audit facile a transmettre : JSON complet, CSV resume par `articleType`, CSV detaille par image, et resume texte lisible. Si Google Drive est monte, les fichiers sont aussi copies dans Drive.


In [ ]:
from datetime import datetime, timezone
import shutil

REPORT_DIR = CONTENT_ARTIFACT_DIR / 'fashion_v1_dataset_audit'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

summary_export = summary.copy()
image_audit_export = audit_df.copy()
image_audit_export['image_path'] = image_audit_export['image_path'].astype(str)
image_audit_export['proposed_product_type_v0'] = image_audit_export['articleType'].map(proposed_product_type)
image_audit_export['proposed_canonical_category'] = image_audit_export['articleType'].map(proposed_canonical_category)

product_type_summary = (
    summary_export.dropna(subset=['proposed_product_type_v0'])
    .groupby(['proposed_product_type_v0', 'proposed_canonical_category'], dropna=False)
    .agg(
        article_type_count=('articleType', 'nunique'),
        metadata_row_count=('metadata_row_count', 'sum'),
        present_image_count=('present_image_count', 'sum'),
        readable_image_count=('readable_image_count', 'sum'),
    )
    .reset_index()
    .sort_values('readable_image_count', ascending=False)
)

missing_values = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
}).sort_values('missing_count', ascending=False)

def series_counts(column, limit=None):
    counts = df[column].value_counts(dropna=False)
    if limit is not None:
        counts = counts.head(limit)
    return {str(key): int(value) for key, value in counts.items()}

report = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'project_dir': str(PROJECT_DIR),
    'metadata_csv': str(STYLES_CSV),
    'image_dir': str(IMAGE_DIR),
    'class_config_path': str(CLASS_CONFIG_PATH),
    'class_config': class_config,
    'dataset_shape': {'rows': int(df.shape[0]), 'columns': int(df.shape[1])},
    'columns': list(df.columns),
    'missing_values': missing_values.reset_index(names='column').to_dict(orient='records'),
    'article_type_distribution_top_80': series_counts('articleType', limit=80),
    'master_category_distribution': series_counts('masterCategory') if 'masterCategory' in df.columns else {},
    'sub_category_distribution': series_counts('subCategory') if 'subCategory' in df.columns else {},
    'image_quality': {
        'metadata_rows': int(len(audit_df)),
        'present_images': int(audit_df['image_present'].sum()),
        'missing_images': int((~audit_df['image_present']).sum()),
        'readable_images': int(audit_df['image_readable'].sum()),
        'unreadable_or_corrupt_images': int((audit_df['image_present'] & ~audit_df['image_readable']).sum()),
    },
    'article_type_audit': summary_export.to_dict(orient='records'),
    'product_type_summary': product_type_summary.to_dict(orient='records'),
}

summary_csv = REPORT_DIR / 'fashion_v1_article_type_audit.csv'
image_audit_csv = REPORT_DIR / 'fashion_v1_image_file_audit.csv'
product_type_csv = REPORT_DIR / 'fashion_v1_product_type_summary.csv'
report_json = REPORT_DIR / 'fashion_v1_dataset_audit_report.json'
report_txt = REPORT_DIR / 'fashion_v1_dataset_audit_summary.txt'

summary_export.to_csv(summary_csv, index=False)
image_audit_export.to_csv(image_audit_csv, index=False)
product_type_summary.to_csv(product_type_csv, index=False)
with report_json.open('w', encoding='utf-8') as handle:
    json.dump(report, handle, ensure_ascii=False, indent=2, default=str)

with report_txt.open('w', encoding='utf-8') as handle:
    handle.write('Fashion Product Images Small - audit dataset V0\n')
    handle.write('=' * 54 + '\n\n')
    handle.write(f"Dataset shape: {df.shape}\n")
    handle.write(f"Styles CSV: {STYLES_CSV}\n")
    handle.write(f"Images dir: {IMAGE_DIR}\n")
    handle.write(f"Config status: {class_config.get('status')}\n")
    handle.write(f"Minimum readable images per class: {class_config.get('minimum_readable_images_per_class')}\n\n")
    handle.write('Image quality\n')
    for key, value in report['image_quality'].items():
        handle.write(f"- {key}: {value}\n")
    handle.write('\nProduct type summary\n')
    handle.write(product_type_summary.to_string(index=False))
    handle.write('\n\nArticleType audit top 80 by readable images\n')
    handle.write(summary_export.sort_values('readable_image_count', ascending=False).head(80).to_string(index=False))
    handle.write('\n')

exported_files = [summary_csv, product_type_csv, image_audit_csv, report_json, report_txt]
drive_report_dir = None
if 'DRIVE_ROOT' in globals() and DRIVE_ROOT.exists():
    drive_report_dir = DRIVE_ROOT / 'fit-outfit-advisor' / 'reports' / 'fashion_v1_dataset_audit'
    drive_report_dir.mkdir(parents=True, exist_ok=True)
    for file_path in exported_files:
        shutil.copy2(file_path, drive_report_dir / file_path.name)

print('Fichiers exportes dans Colab :')
for file_path in exported_files:
    print(f'- {file_path}')
if drive_report_dir:
    print(f'Copies Drive : {drive_report_dir}')

print('\nResume par product_type_v0 propose :')
display(product_type_summary)
print('\nTop ArticleType par images lisibles :')
display(summary_export.sort_values('readable_image_count', ascending=False).head(80))


## 11. Apercu visuel d'images reelles


In [ ]:
import matplotlib.pyplot as plt

sample_df = audit_df[audit_df['image_readable']].copy()
if sample_df.empty:
    raise ValueError('Aucune image lisible pour afficher un apercu visuel.')

sample_df['proposed_product_type_v0'] = sample_df['articleType'].map(proposed_product_type)
sample_df['proposed_canonical_category'] = sample_df['articleType'].map(proposed_canonical_category)
sample_df = sample_df.sample(n=min(12, len(sample_df)), random_state=42)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, (_, row) in zip(axes.flatten(), sample_df.iterrows()):
    image = Image.open(row['image_path']).convert('RGB')
    ax.imshow(image)
    ax.set_title(f"{row['articleType']} -> {row['proposed_product_type_v0']} -> {row['proposed_canonical_category']}", fontsize=9)
    ax.axis('off')
for ax in axes.flatten()[len(sample_df):]:
    ax.axis('off')
plt.tight_layout()
plt.show()


## 12. Validation explicite avant entrainement

Cette cellule bloque seulement si la configuration est encore en brouillon. L'entrainement reste une action volontaire dans les cellules suivantes.


In [ ]:
if class_config.get('status') == 'draft_requires_dataset_inspection':
    raise RuntimeError(
        'ARRET AVANT ENTRAINEMENT : inspecte le tableau final, puis renseigne '
        'config/fashion_v1_classes.json avec les articleType retenus et le seuil minimal.'
    )

print('Configuration validee detectee. Tu peux lancer les cellules d entrainement Fashion V1 si tu le souhaites.')


## 13. Dry-run du pipeline d'entrainement

Cette cellule verifie le dataset, la configuration, les images lisibles et les classes retenues sans entrainer de modele.


In [ ]:
training_env = os.environ.copy()
training_env['PYTHONPATH'] = f"{PROJECT_DIR}{os.pathsep}{training_env.get('PYTHONPATH', '')}".rstrip(os.pathsep)

subprocess.run(
    [
        sys.executable,
        '-m',
        'src.training.train_fashion_model_v1',
        '--metadata-csv',
        str(STYLES_CSV),
        '--image-dir',
        str(IMAGE_DIR),
        '--dry-run',
    ],
    cwd=str(PROJECT_DIR),
    env=training_env,
    check=True,
)


## 14. Entrainer Fashion V1 experimental

Cette cellule entraine `simple_cnn` et `mobilenet_v2`, selectionne l'experience sur validation uniquement, puis evalue le test une seule fois apres selection. Les artefacts restent experimentaux dans `models/fashion_v1/`.


In [ ]:
EPOCHS = 12
BATCH_SIZE = 32
IMAGE_SIZE = 224
PATIENCE = 3

subprocess.run(
    [
        sys.executable,
        '-m',
        'src.training.train_fashion_model_v1',
        '--metadata-csv',
        str(STYLES_CSV),
        '--image-dir',
        str(IMAGE_DIR),
        '--architecture',
        'both',
        '--epochs',
        str(EPOCHS),
        '--batch-size',
        str(BATCH_SIZE),
        '--image-size',
        str(IMAGE_SIZE),
        '--patience',
        str(PATIENCE),
    ],
    cwd=str(PROJECT_DIR),
    env=training_env,
    check=True,
)


## 15. Verifier et copier les artefacts Fashion V1


In [ ]:
import shutil

FASHION_V1_DIR = PROJECT_DIR / 'models' / 'fashion_v1'
EXPECTED_ARTIFACTS = [
    'fashion_model.keras',
    'label_encoder.joblib',
    'metadata.json',
    'metrics.json',
    'confusion_matrix_raw.png',
    'confusion_matrix_normalized.png',
    'training_history.png',
    'sample_predictions.png',
]

print('Artefacts Fashion V1 :')
for artifact_name in EXPECTED_ARTIFACTS:
    artifact_path = FASHION_V1_DIR / artifact_name
    status = 'OK' if artifact_path.exists() else 'ABSENT'
    size = artifact_path.stat().st_size if artifact_path.exists() else 0
    print(f'{status:<7} {artifact_name} {size} bytes')

metrics_path = FASHION_V1_DIR / 'metrics.json'
metadata_path = FASHION_V1_DIR / 'metadata.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('\nSelected experiment:', metrics.get('selected_experiment'))
    print('Reason:', metrics.get('reason_for_selection'))
    print('Validation metrics:')
    for name, values in metrics.get('validation_metrics', {}).items():
        print(name, {key: round(values[key], 4) for key in ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1']})
    selected_test = metrics.get('test_metrics', {}).get('selected_experiment', {})
    print('Test selected:', {key: round(selected_test[key], 4) for key in ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1'] if key in selected_test})
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    print('\nModel status:', metadata.get('model_status'))
    print('Promotable to Streamlit:', metadata.get('promotable_to_streamlit'))

drive_artifact_dir = DRIVE_ROOT / 'fit-outfit-advisor' / 'artifacts' / 'fashion_v1'
drive_artifact_dir.mkdir(parents=True, exist_ok=True)
for artifact_name in EXPECTED_ARTIFACTS:
    artifact_path = FASHION_V1_DIR / artifact_name
    if artifact_path.exists():
        shutil.copy2(artifact_path, drive_artifact_dir / artifact_name)
print(f'Copies Drive : {drive_artifact_dir}')


## 16. Analyser les seuils de confiance Fashion V1

Cette cellule ne reentraine pas le modele. Elle reconstruit les memes splits, selectionne un seuil de confiance sur validation uniquement, puis evalue le test une seule fois au seuil retenu ou diagnostique.


In [ ]:
ABSTENTION_REPORT = PROJECT_DIR / 'reports' / 'fashion_v1_abstention.json'

subprocess.run(
    [
        sys.executable,
        '-m',
        'src.analysis.analyze_fashion_v1_abstention',
        '--metadata-csv',
        str(STYLES_CSV),
        '--image-dir',
        str(IMAGE_DIR),
        '--artifact-dir',
        str(FASHION_V1_DIR),
        '--output',
        str(ABSTENTION_REPORT),
        '--batch-size',
        str(BATCH_SIZE),
        '--image-size',
        str(IMAGE_SIZE),
    ],
    cwd=str(PROJECT_DIR),
    env=training_env,
    check=True,
)

report = json.loads(ABSTENTION_REPORT.read_text(encoding='utf-8'))
selection = report['threshold_selection']
test_eval = report['test_evaluation_at_selected_threshold']
print('Selection:', selection['reason'])
print('Seuil selectionne:', selection.get('selected_threshold'))
print('Seuil diagnostic:', selection.get('diagnostic_best_threshold'))
print('Coverage test:', round(test_eval['coverage'], 4))
print('Unknown rate test:', round(test_eval['unknown_rate'], 4))
print('Accuracy non-unknown test:', round(test_eval['accuracy_non_unknown'], 4))
print('Macro F1 non-unknown test:', round(test_eval['macro_f1_non_unknown'], 4))

drive_report_dir = DRIVE_ROOT / 'fit-outfit-advisor' / 'reports'
drive_report_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(ABSTENTION_REPORT, drive_report_dir / ABSTENTION_REPORT.name)
print(f'Rapport abstention copie : {drive_report_dir / ABSTENTION_REPORT.name}')
